In [29]:
import pandas as pd

In [30]:
df = pd.read_csv('repair_invoices_v3.csv')

In [31]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11793 entries, 0 to 11792
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   repair_id               11793 non-null  int64  
 1   claim_id                11793 non-null  int64  
 2   vehicle_id              11793 non-null  int64  
 3   repair_shop_contact_id  11793 non-null  int64  
 4   estimate_amount         11793 non-null  float64
 5   approved_amount         11793 non-null  float64
 6   invoice_amount          11793 non-null  float64
 7   paid_amount             11793 non-null  float64
 8   repair_start_date       11793 non-null  object 
 9   repair_end_date         11793 non-null  object 
 10  repair_status           11793 non-null  object 
 11  make                    11793 non-null  object 
 12  model                   11793 non-null  object 
 13  vehicle_year            11793 non-null  int64  
 14  claim_status            11793 non-null

,repair_id,claim_id,vehicle_id,repair_shop_contact_id,estimate_amount,approved_amount,invoice_amount,paid_amount,vehicle_year,liability_percentage
count,11793.000000,11793.000000,11793.000000,11793.000000,11793.000000,11793.000000,11793.000000,11793.000000,11793.000000,11793.000000
mean,5897.000000,4236.902485,3936.972017,110.101925,3008.080655,2820.672724,3143.908777,2781.082423,2016.558891,82.421812
std,3404.490197,2446.801812,2242.965670,63.080687,2528.266856,2311.498164,2820.068021,2286.700377,5.192539,16.271902
min,1.000000,1.000000,1.000000,1.000000,507.570000,457.020000,426.320000,426.730000,2008.000000,6.317839
25%,2949.000000,2116.000000,2007.000000,55.000000,1313.380000,1239.250000,1335.480000,1223.558359,2012.000000,71.828769
50%,5897.000000,4241.000000,3992.000000,110.000000,2228.270000,2099.660000,2278.894727,2067.750000,2017.000000,85.118144
75%,8845.000000,6347.000000,5879.000000,164.000000,3798.340000,3556.660000,3943.790000,3508.850000,2021.000000,98.552727
max,11793.000000,8468.000000,7800.000000,220.000000,46722.112563,24446.380000,47701.262952,22686.460000,2025.000000,100.000000


In [32]:
df.head()

,repair_id,claim_id,vehicle_id,repair_shop_contact_id,estimate_amount,approved_amount,invoice_amount,paid_amount,repair_start_date,repair_end_date,...,incident_state,incident_country,total_loss,liability_percentage,exposure_type,coverage_type,exposure_status,is_repeat_vehicle,is_synthetic_anomaly,anomaly_type
0,1,1,5749,217,883.00,859.88,821.28,779.60,2025-11-16,2025-11-18,...,CA,USA,False,71.924662,Third Party Property Damage,Liability,Open,False,False,NaN
1,2,2,1818,152,1659.62,1620.74,1627.69,1621.15,2026-02-23,2026-03-02,...,TX,USA,False,54.300753,Third Party Property Damage,Liability,Closed,False,False,NaN
2,3,3,5113,64,1044.97,1075.69,1146.53,1051.40,2025-05-21,2025-05-28,...,IL,USA,False,35.255929,First Party Collision,Collision,Closed,False,False,NaN
3,4,3,5113,64,1325.40,1235.23,1448.16,1182.52,2025-05-21,2025-05-28,...,IL,USA,False,35.255929,First Party Collision,Collision,Closed,False,False,NaN
4,5,4,5558,1,890.20,934.05,754.12,753.83,2025-03-28,2025-04-01,...,IL,USA,False,100.000000,First Party Collision,Comprehensive,Open,False,False,NaN


In [33]:
print(len(df))
print(df['repair_id'].nunique())

11793
11793


In [34]:
df['damage_severity'].value_counts()

damage_severity
Minor         4795
Moderate      4077
Severe        2151
Total Loss     770
Name: count, dtype: int64

In [35]:
df = df[df['exposure_type'] == 'Third Party Property Damage'].copy()

In [36]:
df.dtypes

repair_id                   int64
claim_id                    int64
vehicle_id                  int64
repair_shop_contact_id      int64
estimate_amount           float64
approved_amount           float64
invoice_amount            float64
paid_amount               float64
repair_start_date          object
repair_end_date            object
repair_status              object
make                       object
model                      object
vehicle_year                int64
claim_status               object
claim_severity             object
loss_cause                 object
jurisdiction               object
incident_type              object
damage_severity            object
incident_state             object
incident_country           object
total_loss                   bool
liability_percentage      float64
exposure_type              object
coverage_type              object
exposure_status            object
is_repeat_vehicle            bool
is_synthetic_anomaly         bool
anomaly_type  

In [37]:
df["repair_start_date"] = pd.to_datetime(df["repair_start_date"])
df["repair_end_date"] = pd.to_datetime(df["repair_end_date"])

In [38]:
claims_per_vehicle = df.groupby("vehicle_id")["claim_id"].nunique().rename("vehicle_claim_count")

In [39]:
df = df.merge(claims_per_vehicle, on="vehicle_id", how="left")

In [40]:
df.dtypes

repair_id                          int64
claim_id                           int64
vehicle_id                         int64
repair_shop_contact_id             int64
estimate_amount                  float64
approved_amount                  float64
invoice_amount                   float64
paid_amount                      float64
repair_start_date         datetime64[ns]
repair_end_date           datetime64[ns]
repair_status                     object
make                              object
model                             object
vehicle_year                       int64
claim_status                      object
claim_severity                    object
loss_cause                        object
jurisdiction                      object
incident_type                     object
damage_severity                   object
incident_state                    object
incident_country                  object
total_loss                          bool
liability_percentage             float64
exposure_type   

In [41]:
df['repair_duration_days'] = (df['repair_end_date'] - df['repair_start_date']).dt.days.clip(lower=1)

In [42]:
df['vehicle_age'] = pd.Timestamp.now().year - df['vehicle_year']

In [43]:
df["invoice_vs_estimate_ratio"] = df["invoice_amount"] / df["estimate_amount"]
df["invoice_vs_approved_ratio"] = df["invoice_amount"] / df["approved_amount"]


In [44]:
df

,repair_id,claim_id,vehicle_id,repair_shop_contact_id,estimate_amount,approved_amount,invoice_amount,paid_amount,repair_start_date,repair_end_date,...,coverage_type,exposure_status,is_repeat_vehicle,is_synthetic_anomaly,anomaly_type,vehicle_claim_count,repair_duration_days,vehicle_age,invoice_vs_estimate_ratio,invoice_vs_approved_ratio
0,1,1,5749,217,883.00,859.88,821.280000,779.60,2025-11-16,2025-11-18,...,Liability,Open,False,False,NaN,1,2,13,0.930102,0.955110
1,2,2,1818,152,1659.62,1620.74,1627.690000,1621.15,2026-02-23,2026-03-02,...,Liability,Closed,False,False,NaN,1,7,6,0.980761,1.004288
2,7,6,4529,175,1013.25,844.58,917.770000,820.13,2026-04-06,2026-04-12,...,Liability,Open,False,False,NaN,1,6,3,0.905769,1.086658
3,8,6,4529,175,1231.42,1213.63,1417.890000,1211.47,2026-04-06,2026-04-12,...,Liability,Open,False,False,NaN,1,6,3,1.151427,1.168305
4,9,7,7329,180,2946.86,3039.04,2827.410000,2918.91,2025-11-23,2025-11-29,...,Liability,Closed,False,False,NaN,1,6,4,0.959465,0.930363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6451,11782,4334,3048,153,844.03,857.15,936.699292,855.36,2026-05-15,2026-05-16,...,Liability,Open,True,True,duplicate_billing,4,1,6,1.109794,1.092807
6452,11783,3732,5904,84,3660.92,3451.44,3544.885344,3462.86,2025-02-07,2025-02-22,...,Liability,Open,False,True,duplicate_billing,1,15,16,0.968305,1.027074
6453,11785,2642,6525,212,2583.02,2347.32,2859.519975,2245.39,2025-09-23,2025-10-01,...,Liability,Closed,True,True,duplicate_billing,4,8,12,1.107045,1.218206
6454,11786,60,7772,51,3275.08,3093.64,3017.979816,3031.68,2026-03-08,2026-03-26,...,Liability,Closed,False,True,duplicate_billing,1,18,17,0.921498,0.975543


In [45]:
df["duplicate_flag"] = False

for claim_id, group in df.groupby("claim_id"):
    if len(group) < 2:
        continue
    amounts = group["invoice_amount"].values
    idxs = group.index.values
    for i in range(len(amounts)):
        for j in range(i + 1, len(amounts)):
            if abs(amounts[i] - amounts[j]) / max(amounts[i], amounts[j]) < 0.03:
                df.loc[idxs[i], "duplicate_flag"] = True
                df.loc[idxs[j], "duplicate_flag"] = True

In [46]:
df.columns

Index(['repair_id', 'claim_id', 'vehicle_id', 'repair_shop_contact_id',
       'estimate_amount', 'approved_amount', 'invoice_amount', 'paid_amount',
       'repair_start_date', 'repair_end_date', 'repair_status', 'make',
       'model', 'vehicle_year', 'claim_status', 'claim_severity', 'loss_cause',
       'jurisdiction', 'incident_type', 'damage_severity', 'incident_state',
       'incident_country', 'total_loss', 'liability_percentage',
       'exposure_type', 'coverage_type', 'exposure_status',
       'is_repeat_vehicle', 'is_synthetic_anomaly', 'anomaly_type',
       'vehicle_claim_count', 'repair_duration_days', 'vehicle_age',
       'invoice_vs_estimate_ratio', 'invoice_vs_approved_ratio',
       'duplicate_flag'],
      dtype='object')

In [47]:
df = df.drop(columns=['incident_country','exposure_type','total_loss','jurisdiction','incident_type','coverage_type'])

In [50]:
df

,repair_id,claim_id,vehicle_id,repair_shop_contact_id,estimate_amount,approved_amount,invoice_amount,paid_amount,repair_start_date,repair_end_date,...,exposure_status,is_repeat_vehicle,is_synthetic_anomaly,anomaly_type,vehicle_claim_count,repair_duration_days,vehicle_age,invoice_vs_estimate_ratio,invoice_vs_approved_ratio,duplicate_flag
0,1,1,5749,217,883.00,859.88,821.280000,779.60,2025-11-16,2025-11-18,...,Open,False,False,NaN,1,2,13,0.930102,0.955110,True
1,2,2,1818,152,1659.62,1620.74,1627.690000,1621.15,2026-02-23,2026-03-02,...,Closed,False,False,NaN,1,7,6,0.980761,1.004288,False
2,7,6,4529,175,1013.25,844.58,917.770000,820.13,2026-04-06,2026-04-12,...,Open,False,False,NaN,1,6,3,0.905769,1.086658,False
3,8,6,4529,175,1231.42,1213.63,1417.890000,1211.47,2026-04-06,2026-04-12,...,Open,False,False,NaN,1,6,3,1.151427,1.168305,False
4,9,7,7329,180,2946.86,3039.04,2827.410000,2918.91,2025-11-23,2025-11-29,...,Closed,False,False,NaN,1,6,4,0.959465,0.930363,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6451,11782,4334,3048,153,844.03,857.15,936.699292,855.36,2026-05-15,2026-05-16,...,Open,True,True,duplicate_billing,4,1,6,1.109794,1.092807,True
6452,11783,3732,5904,84,3660.92,3451.44,3544.885344,3462.86,2025-02-07,2025-02-22,...,Open,False,True,duplicate_billing,1,15,16,0.968305,1.027074,True
6453,11785,2642,6525,212,2583.02,2347.32,2859.519975,2245.39,2025-09-23,2025-10-01,...,Closed,True,True,duplicate_billing,4,8,12,1.107045,1.218206,True
6454,11786,60,7772,51,3275.08,3093.64,3017.979816,3031.68,2026-03-08,2026-03-26,...,Closed,False,True,duplicate_billing,1,18,17,0.921498,0.975543,True


In [51]:
output = "repair_invoice_clean.csv"

In [52]:
df.to_csv(output, index=False)